# CloudDenseNet — DenseNet121 Transfer Learning on Filtered NASA GLOBE

Replication of *CloudDenseNet: Lightweight Ground-Based Cloud Classification*
(Li et al., Sensors 2023).

**What this notebook does:**
- Loads filtered NASA GLOBE images via the ResNet18 cloud-filter CSV
- Fine-tunes DenseNet121 (ImageNet weights) using the paper's exact training schedule:
  a custom **TopBlock** head, **Focal Loss**, and **5-phase gradual unfreezing**

**Data pipeline** (cells 1–8): unchanged from the baseline notebook.

**Training schedule** (cells 9–14): replicates paper §4:

| Phase | Layers unfrozen | Head LR | Backbone LR | Epochs |
|-------|----------------|---------|-------------|--------|
| 1 | TopBlock only | 1e-4 | — | 10 |
| 2 | TopBlock only | 1e-5 | — | 10 |
| 3 | TopBlock + DenseBlock3 onwards | 1e-5 | 5e-5 | 20 |
| 4 | TopBlock + DenseBlock3 onwards | 1e-5 | 1e-5 | 20 |
| 5 | All layers | 1e-6 | 1e-6 | 20 |

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter

CSV_PATH    = Path("../resources/cloud-images/NASA_GLOBE_CD/cloud_filter_results_1.csv")
IMAGES_ROOT = Path("../resources/cloud-images/NASA_GLOBE_CD/downloaded_images")

THRESHOLDS = {
    "Ac": 0.99,
    "As": 0.99,
    "Cb": 0.99,
    "Cc": 0.99,
    "Ci": 0.99,
    "Cs": 0.99,
    "Cu": 0.99,
    "Ns": 0.99,
    "Sc": 0.99,
    "St": 0.99,
}

def index_labeled_images_filtered(csv_path, thresholds, images_root, max_per_class=3000):
    df = pd.read_csv(csv_path)
    df = df.rename(columns={"cloud_conf": "head2_conf", "head1_class": "head1_pred"})
    labeled_images = {}
    for folder, thresh in thresholds.items():
        clean = df[(df["folder"] == folder) & (df["head2_conf"] >= thresh)]
        clean = clean.sample(min(max_per_class, len(clean)), random_state=42)
        for _, row in clean.iterrows():
            labeled_images[row["filename"]] = {
                "label": folder,
                "path":  str(Path(images_root) / folder / row["filename"]),
            }
    return labeled_images

labeled_images = index_labeled_images_filtered(CSV_PATH, THRESHOLDS, IMAGES_ROOT)

counts = Counter(v["label"] for v in labeled_images.values())
for folder in sorted(counts):
    print(f"  {folder:6s}  {counts[folder]:,}")
print(f"  {'TOTAL':6s}  {sum(counts.values()):,}")

  Ac      3,000
  As      3,000
  Cb      3,000
  Cc      3,000
  Ci      3,000
  Cs      3,000
  Cu      3,000
  Ns      3,000
  Sc      3,000
  St      3,000
  TOTAL   30,000


In [2]:
def extract_labels(labeled_images):
    paths, labels = [], []
    for image_name in labeled_images:
        paths.append(labeled_images[image_name]["path"])
        labels.append(labeled_images[image_name]["label"])
    return paths, np.array(labels)

paths, labels = extract_labels(labeled_images)

In [3]:
import torch

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"device: {device}")

device: mps


In [4]:
import torchvision.transforms.v2 as T
from torchvision.transforms.v2 import InterpolationMode

IMG_SIZE = (224, 224)
MAX_FRAC = 0.14

border_translation = T.RandomAffine(
    degrees=0,
    translate=(MAX_FRAC, MAX_FRAC),
    interpolation=InterpolationMode.BILINEAR,
    fill=0
)

# T.Lambda is fine locally with num_workers=0
wrap_translation = T.Lambda(lambda x: torch.roll(
    x,
    shifts=(
        int(torch.randint(-int(MAX_FRAC * x.shape[-2]), int(MAX_FRAC * x.shape[-2]) + 1, (1,)).item()),
        int(torch.randint(-int(MAX_FRAC * x.shape[-1]), int(MAX_FRAC * x.shape[-1]) + 1, (1,)).item()),
    ),
    dims=(-2, -1),
))

stacked = T.Compose([
    T.RandomChoice([wrap_translation, border_translation]),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.15),
    T.RandomAffine(
        degrees=20, scale=(0.90, 1.10),
        interpolation=InterpolationMode.BILINEAR, fill=0
    ),
    T.RandomResizedCrop(
        size=IMG_SIZE, scale=(0.80, 1.00), ratio=(0.90, 1.10),
        interpolation=InterpolationMode.BILINEAR
    ),
])

one_of = T.RandomChoice([
    wrap_translation,
    border_translation,
    T.RandomChoice([T.RandomHorizontalFlip(p=1.0), T.RandomVerticalFlip(p=1.0)]),
    T.RandomRotation(degrees=20, interpolation=InterpolationMode.BILINEAR, fill=0),
])

normalize = T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

train_transforms = T.Compose([
    T.Resize(IMG_SIZE, interpolation=InterpolationMode.BILINEAR),
    T.ToImage(),
    T.ToDtype(torch.float32, scale=True),
    T.RandomChoice([stacked, one_of]),
    normalize,
])

eval_transforms = T.Compose([
    T.Resize(IMG_SIZE, interpolation=InterpolationMode.BILINEAR),
    T.ToImage(),
    T.ToDtype(torch.float32, scale=True),
    normalize,
])

In [5]:
import torchvision
import torch.nn as nn

weights = torchvision.models.DenseNet121_Weights.IMAGENET1K_V1
model = torchvision.models.densenet121(weights=weights).to(device)
print("DenseNet121 loaded")

DenseNet121 loaded


In [6]:
from sklearn.preprocessing import LabelEncoder

def encode_labels(labels):
    ordinal_encoder = LabelEncoder()
    encoded_labels = ordinal_encoder.fit_transform(labels)
    return encoded_labels, ordinal_encoder.classes_

encoded_labels, class_names = encode_labels(labels)
print(f"Classes ({len(class_names)}): {class_names}")

Classes (10): ['Ac' 'As' 'Cb' 'Cc' 'Ci' 'Cs' 'Cu' 'Ns' 'Sc' 'St']


In [7]:
from torch.utils.data import Dataset
from sklearn.model_selection import StratifiedShuffleSplit
from PIL import Image

class MyImages(Dataset):
    def __init__(self, paths, encoded_labels, split="train", test_size=0.2, val_size=0.1,
                 random_state=42, transform=None, split_indices=None):
        if split not in {None, "train", "val", "test"}:
            raise ValueError(f"split must be one of {{'None','train','val','test'}}, got {split!r}")

        self.transform = transform
        paths = np.array(list(paths))
        encoded_labels = np.array(list(encoded_labels))

        if split_indices is None:
            sss1 = StratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
            trainval_idx, test_idx = next(sss1.split(paths, encoded_labels))

            val_within_trainval = val_size / (1.0 - test_size)
            sss2 = StratifiedShuffleSplit(n_splits=1, test_size=val_within_trainval, random_state=random_state)
            train_rel_idx, val_rel_idx = next(sss2.split(paths[trainval_idx], encoded_labels[trainval_idx]))

            split_indices = {
                "train": trainval_idx[train_rel_idx],
                "val":   trainval_idx[val_rel_idx],
                "test":  test_idx,
            }

        idx = split_indices[split]
        self.paths = paths[idx].tolist()
        self.encoded_labels = encoded_labels[idx].tolist()

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, self.encoded_labels[idx]


paths_arr  = np.array(list(paths))
labels_arr = np.array(list(encoded_labels))

sss1 = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
trainval_idx, test_idx = next(sss1.split(paths_arr, labels_arr))

sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.1/0.8, random_state=42)
train_rel_idx, val_rel_idx = next(sss2.split(paths_arr[trainval_idx], labels_arr[trainval_idx]))

split_indices = {
    "train": trainval_idx[train_rel_idx],
    "val":   trainval_idx[val_rel_idx],
    "test":  test_idx,
}

train_set = MyImages(paths=paths, encoded_labels=encoded_labels,
                     split="train", transform=train_transforms,
                     split_indices=split_indices)
valid_set = MyImages(paths=paths, encoded_labels=encoded_labels,
                     split="val",   transform=eval_transforms,
                     split_indices=split_indices)
test_set  = MyImages(paths=paths, encoded_labels=encoded_labels,
                     split="test",  transform=eval_transforms,
                     split_indices=split_indices)

print(f"train: {len(train_set):,}  val: {len(valid_set):,}  test: {len(test_set):,}")

train: 21,000  val: 3,000  test: 6,000


In [8]:
from torch.utils.data import DataLoader, WeightedRandomSampler

train_labels_list = [train_set.encoded_labels[i] for i in range(len(train_set))]
class_counts  = np.bincount(train_labels_list)
class_weights = 1.0 / class_counts
sample_weights = torch.tensor([class_weights[l] for l in train_labels_list], dtype=torch.float)

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

# num_workers=0 required locally — T.Lambda in transforms cannot be pickled by worker processes
train_loader = DataLoader(train_set, batch_size=64, sampler=sampler, num_workers=0)
valid_loader = DataLoader(valid_set, batch_size=64, num_workers=0)
test_loader  = DataLoader(test_set,  batch_size=64, num_workers=0)

In [9]:
import torch.nn as nn

n_classes = len(class_names)


class TopBlock(nn.Module):
    """
    Custom classification head from CloudDenseNet (Li et al., Sensors 2023).

    Replaces DenseNet121's single linear classifier with a 2-layer MLP:
      BN → Dropout → Linear(1024, hidden) → ReLU → BN → Dropout → Linear(hidden, n_classes)

    Weights are initialised with LeCun uniform distribution, as specified in the paper.
    """
    def __init__(self, in_features: int, n_classes: int,
                 hidden_dim: int = 512, dropout: float = 0.5):
        super().__init__()
        self.block = nn.Sequential(
            nn.BatchNorm1d(in_features),
            nn.Dropout(p=dropout),
            nn.Linear(in_features, hidden_dim),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(hidden_dim),
            nn.Dropout(p=dropout / 2),
            nn.Linear(hidden_dim, n_classes),
        )
        self._lecun_init()

    def _lecun_init(self):
        """LeCun uniform initialisation for Linear layers (as per paper)."""
        for m in self.block:
            if isinstance(m, nn.Linear):
                nn.init.kaiming_uniform_(m.weight, mode='fan_in', nonlinearity='linear')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.block(x)


# Freeze entire backbone; replace classifier with TopBlock
for param in model.parameters():
    param.requires_grad = False

model.classifier = TopBlock(
    in_features=1024,   # DenseNet121 backbone output channels
    n_classes=n_classes,
).to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Classes ({n_classes}): {class_names}")
print(f"Trainable params: {trainable:,} / {total:,}")

Classes (10): ['Ac' 'As' 'Cb' 'Cc' 'Ci' 'Cs' 'Cu' 'Ns' 'Sc' 'St']
Trainable params: 533,002 / 7,486,858


In [10]:
import torchmetrics


def evaluate_tm(model, data_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            metric.update(model(X_batch), y_batch)
    return metric.compute()


In [11]:
import torch.nn.functional as F


class FocalLoss(nn.Module):
    """
    Focal Loss (Lin et al., 2017).

    Down-weights easy (high-confidence) examples so the model focuses on
    hard, misclassified ones.  gamma=0 reduces to standard cross-entropy.
    Used by the paper in place of cross-entropy.
    """
    def __init__(self, gamma: float = 2.0, reduction: str = 'mean'):
        super().__init__()
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, reduction='none')
        p_t = torch.exp(-ce)
        loss = ((1.0 - p_t) ** self.gamma) * ce
        return loss.mean() if self.reduction == 'mean' else loss.sum()


focal_loss = FocalLoss(gamma=2.0).to(device)
print("FocalLoss ready (gamma=2.0)")

FocalLoss ready (gamma=2.0)


In [12]:
def train_phase(model, optimizer, loss_fn, metric,
                train_loader, valid_loader,
                n_epochs, patience, checkpoint_path, phase_label=''):
    """
    Run one phase of the gradual-unfreeze schedule.

    Keeps frozen BatchNorm layers in eval mode so their running statistics
    are not corrupted; only BN layers with requires_grad=True are trained.
    Best weights are saved to checkpoint_path and restored before returning.
    """
    history   = {'train_losses': [], 'train_metrics': [], 'valid_metrics': []}
    best_val  = 0.0
    no_improve = 0

    for epoch in range(n_epochs):
        # Set correct train/eval mode for frozen vs unfrozen BN layers
        model.eval()
        model.classifier.train()
        for module in model.modules():
            if isinstance(module, (nn.BatchNorm2d, nn.BatchNorm1d)):
                if any(p.requires_grad for p in module.parameters()):
                    module.train()

        total_loss = 0.0
        metric.reset()

        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            y_pred = model(X_batch)
            loss   = loss_fn(y_pred, y_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            metric.update(y_pred, y_batch)

        train_loss = total_loss / len(train_loader)
        train_acc  = metric.compute().item()
        val_acc    = evaluate_tm(model, valid_loader, metric).item()

        history['train_losses'].append(train_loss)
        history['train_metrics'].append(train_acc)
        history['valid_metrics'].append(val_acc)

        star = ' *' if val_acc > best_val else ''
        print(f'[{phase_label}] Epoch {epoch+1}/{n_epochs} | '
              f'loss: {train_loss:.4f} | '
              f'train: {train_acc:.4f} | '
              f'val: {val_acc:.4f}{star}')

        if val_acc > best_val:
            best_val   = val_acc
            no_improve = 0
            torch.save(model.state_dict(), checkpoint_path)
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'  Early stop at epoch {epoch+1} '
                      f'(best val: {best_val:.4f})')
                break

    model.load_state_dict(torch.load(checkpoint_path, weights_only=True))
    return history, best_val


In [13]:
accuracy    = torchmetrics.Accuracy(task='multiclass', num_classes=n_classes).to(device)
CKPT        = 'best_cloudensenet_14.pt'
all_history = []

# ── Phase 1 — TopBlock only, LR = 1e-4, 10 epochs ───────────────────────────
for param in model.parameters():
    param.requires_grad = False
for param in model.classifier.parameters():
    param.requires_grad = True

opt = torch.optim.AdamW(model.classifier.parameters(), lr=1e-4, weight_decay=1e-4)
h, _ = train_phase(model, opt, focal_loss, accuracy,
                   train_loader, valid_loader,
                   n_epochs=10, patience=10,
                   checkpoint_path=CKPT, phase_label='Phase 1 (head, 1e-4)')
all_history.append(h)

# ── Phase 2 — TopBlock only, LR = 1e-5, 10 epochs ───────────────────────────
opt = torch.optim.AdamW(model.classifier.parameters(), lr=1e-5, weight_decay=1e-4)
h, _ = train_phase(model, opt, focal_loss, accuracy,
                   train_loader, valid_loader,
                   n_epochs=10, patience=10,
                   checkpoint_path=CKPT, phase_label='Phase 2 (head, 1e-5)')
all_history.append(h)

# ── Phase 3 — unfreeze DenseBlock3 onwards, LR = 5e-5 backbone, 20 epochs ──
# 'Fine-tune from the 3rd block' (paper §4) = denseblock3, transition3,
# denseblock4, and norm5 in PyTorch's DenseNet121 naming.
unfreeze_from = ['features.denseblock3', 'features.transition3',
                 'features.denseblock4', 'features.norm5']
for name, param in model.named_parameters():
    if any(name.startswith(p) for p in unfreeze_from):
        param.requires_grad = True

backbone_params = [p for n, p in model.named_parameters()
                   if p.requires_grad and not n.startswith('classifier')]
head_params     = list(model.classifier.parameters())

opt = torch.optim.AdamW([
    {'params': head_params,     'lr': 1e-5},
    {'params': backbone_params, 'lr': 5e-5},
], weight_decay=1e-4)
h, _ = train_phase(model, opt, focal_loss, accuracy,
                   train_loader, valid_loader,
                   n_epochs=20, patience=10,
                   checkpoint_path=CKPT,
                   phase_label='Phase 3 (denseblock3+, 5e-5)')
all_history.append(h)

# ── Phase 4 — same unfrozen scope, LR = 1e-5, 20 epochs ────────────────────
opt = torch.optim.AdamW([
    {'params': head_params,     'lr': 1e-5},
    {'params': backbone_params, 'lr': 1e-5},
], weight_decay=1e-4)
h, _ = train_phase(model, opt, focal_loss, accuracy,
                   train_loader, valid_loader,
                   n_epochs=20, patience=10,
                   checkpoint_path=CKPT,
                   phase_label='Phase 4 (denseblock3+, 1e-5)')
all_history.append(h)

# ── Phase 5 — all layers, LR = 1e-6, 20 epochs ──────────────────────────────
for param in model.parameters():
    param.requires_grad = True

opt = torch.optim.AdamW(model.parameters(), lr=1e-6, weight_decay=1e-4)
h, _ = train_phase(model, opt, focal_loss, accuracy,
                   train_loader, valid_loader,
                   n_epochs=20, patience=10,
                   checkpoint_path=CKPT,
                   phase_label='Phase 5 (all, 1e-6)')
all_history.append(h)

test_acc = evaluate_tm(model, test_loader, accuracy)
print(f'\nFinal test accuracy: {test_acc:.4f}')


[Phase 1 (head, 1e-4)] Epoch 1/10 | loss: 2.0371 | train: 0.1763 | val: 0.2543 *
[Phase 1 (head, 1e-4)] Epoch 2/10 | loss: 1.8160 | train: 0.2173 | val: 0.2773 *
[Phase 1 (head, 1e-4)] Epoch 3/10 | loss: 1.7043 | train: 0.2345 | val: 0.2767
[Phase 1 (head, 1e-4)] Epoch 4/10 | loss: 1.6668 | train: 0.2360 | val: 0.2790 *
[Phase 1 (head, 1e-4)] Epoch 5/10 | loss: 1.6265 | train: 0.2469 | val: 0.2887 *
[Phase 1 (head, 1e-4)] Epoch 6/10 | loss: 1.5947 | train: 0.2533 | val: 0.2840
[Phase 1 (head, 1e-4)] Epoch 7/10 | loss: 1.5714 | train: 0.2518 | val: 0.2983 *
[Phase 1 (head, 1e-4)] Epoch 8/10 | loss: 1.5674 | train: 0.2533 | val: 0.3060 *
[Phase 1 (head, 1e-4)] Epoch 9/10 | loss: 1.5458 | train: 0.2619 | val: 0.3147 *
[Phase 1 (head, 1e-4)] Epoch 10/10 | loss: 1.5444 | train: 0.2506 | val: 0.3037
[Phase 2 (head, 1e-5)] Epoch 1/10 | loss: 1.5486 | train: 0.2627 | val: 0.3167 *
[Phase 2 (head, 1e-5)] Epoch 2/10 | loss: 1.5236 | train: 0.2653 | val: 0.3177 *
[Phase 2 (head, 1e-5)] Epoch 3/10

KeyboardInterrupt: 

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

colors = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple']
labels = [
    'Phase 1 (head, 1e-4)',
    'Phase 2 (head, 1e-5)',
    'Phase 3 (block3+, 5e-5)',
    'Phase 4 (block3+, 1e-5)',
    'Phase 5 (all, 1e-6)',
]

offset = 0
for h, label, color in zip(all_history, labels, colors):
    xs = list(range(offset, offset + len(h['train_losses'])))
    axes[0].plot(xs, h['train_losses'], color=color, label=label)
    axes[1].plot(xs, h['train_metrics'], color=color, linestyle='--', alpha=0.5)
    axes[1].plot(xs, h['valid_metrics'],  color=color, label=label)
    # shade alternate phases for readability
    if len(all_history) > 1:
        for ax in axes:
            ax.axvline(x=offset, color='grey', linewidth=0.5, linestyle=':')
    offset += len(h['train_losses'])

axes[0].set_title('Focal Loss — all 5 phases')
axes[0].set_xlabel('Epoch (cumulative)')
axes[0].set_ylabel('Focal Loss')
axes[0].legend(fontsize=8)

axes[1].set_title('Accuracy — dashed=train, solid=val')
axes[1].set_xlabel('Epoch (cumulative)')
axes[1].set_ylabel('Accuracy')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()
